# 5G Network Intelligence — Chronos-2 Adaptation Evaluation

## Research question
Does native multivariate LoRA improve DLPRB forecasting over both the unchanged synthetic-pretrained model and the strongest saved development controls? Training loss is not a forecasting accuracy metric.


## Temporal protocol and selection
Fit/stop/score cutoffs are 335/383/443 and 395/443/503. Context: 168 hours; forecast horizons: 1, 6 and 24 hours. All original beams are evaluated. Select steps 100/300/1000 using stopping MAE only, breaking ties toward the earlier step. Freeze both choices before scoring. Hours 504 onward are excluded. The previously observed holdout is never reused.


In [ ]:
# Safe review default: no training, inference or checkpoint selection.
RUN_EVALUATION = False
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
PROJECT = Path.cwd()


## Artifact provenance and debugging
The separate hosted evaluator verifies base-weight hashes, every training artifact, LoRA configuration, optimizer-step identities and runtime versions. Inference caches bind the model and exact causal history. A completed training job does not imply checkpoint selection or evaluation has completed.


In [ ]:
# Inspect the recorded submission; this is not a completion claim.
submission = PROJECT / 'reports/chronos2_lora_cloud_submission.json'
if submission.exists():
    display(json.loads(submission.read_text(encoding='utf-8')))


## Hosted evaluation entry point
This stage performs inference and stopping-only checkpoint selection, never fitting. Use a verified completed training output and the pinned base model in a hosted GPU runtime. Durable outputs belong on Drive or Kaggle Output.


In [ ]:
if RUN_EVALUATION:
    import subprocess, sys
    if 'MODEL_ROOT' not in globals() or 'TRAINED_ROOT' not in globals():
        raise RuntimeError('Attach and verify base model and completed training artifacts first')
    subprocess.run([sys.executable, '-u', '-m', 'scripts.evaluate_chronos2_lora',
                    '--model-root', str(MODEL_ROOT), '--trained-root', str(TRAINED_ROOT)], check=True)
else:
    print('Review mode: checkpoint selection and inference skipped.')


## Stopping evidence
MAE is measured in original DLPRB units. This table explains checkpoint choice; it is separate from scoring performance.


In [ ]:
stopping = PROJECT / 'reports/chronos2_lora_stopping.csv'
if stopping.exists():
    display(pd.read_csv(stopping))
else:
    print('No verified stopping results available yet.')


## Development performance
Compare unchanged multivariate zero-shot and selected LoRA on identical frozen origins and observed target masks. MAE uses original DLPRB units; WAPE is sum absolute error / sum absolute actual values. Coverage and zero false-positive rates expose changes hidden by aggregate error.


In [ ]:
scores = PROJECT / 'reports/chronos2_lora_development.csv'
if scores.exists():
    results = pd.read_csv(scores)
    results['WAPE (%)'] = 100 * results['wape']
    results['Zero false positives (%)'] = 100 * results['zero_fp_rate']
    display(results[['fold', 'stage', 'horizon', 'mae', 'WAPE (%)',
                     'n_valid', 'Zero false positives (%)']])
else:
    print('No verified development metrics available yet; no accuracy claim.')


## Acceptance and limitations
Compare with the saved LightGBM hurdle and MLP-B/hurdle controls on the same masks. Require no horizon regression and at least 2% mean relative MAE improvement over the strong control before further seed and uncertainty checks. Report negative results. A single seed and reused development data cannot establish independent generalization. Synthetic-only pretraining is the publisher's claim, not an independent corpus audit. WAPE <10% is unproven. No automatic production promotion or new holdout evaluation.
